<a href="https://colab.research.google.com/github/teamepic043/Interactive-Campus-Info-Chatbot-AI-Agent/blob/main/Copy_of_commit10c_langchain_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Campus Query Agent — ReAct + Sitemap + Location + Contact Info + Smart Search

```
ONE TIME per college
  build_sitemap()  crawls 2 levels (HTTP only, no LLM)
                   LLM classifies all links into topic map  (1 LLM call)
                   saves sitemap_{domain}.json to disk

PER QUERY
  ReAct Agent      lookup_sitemap(topic)       reads disk, free
                   fetch_page(url)             cached HTTP
                   get_page_links(url)         cached HTTP
                   read_pdf(url)               HTTP + pypdf
                   search_web(query)           DuckDuckGo fallback
                   search_topic(topic)         NEW - structured topic search
                   get_college_location()      LLM address extraction
                   get_contact_info()          regex + LLM contact extraction
                   agent reasons until it has enough to answer
```
NEW: Smart Search tab with category buttons for Hostel, Transport,
Academic Calendar, Fees, Placements, Admissions, Facilities, Research, Events.


In [ ]:
# Cell 1 - Install
!pip install langchain langchain-google-genai langchain-community langgraph beautifulsoup4 requests pypdf gradio duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# Cell 2 - Imports
import io
import json
import logging
import os
import re
from datetime import datetime
from urllib.parse import urljoin, urlparse
from typing import List, Optional

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from IPython.display import Markdown, display
from duckduckgo_search import DDGS

logging.getLogger('pypdf').setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field


In [ ]:
# Cell 3 - API Key
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('Gemini1')


In [ ]:
# Cell 4 - LLM
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    temperature=0
)


In [ ]:
# Cell 5 - Raw Web Scraping Utilities

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/117.0.0.0 Safari/537.36'
}

def _raw_fetch_text(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return BeautifulSoup(r.text, 'html.parser').get_text(separator=' ', strip=True)

def _raw_fetch_links(url: str) -> list:
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, 'html.parser')
    results = []
    for a in soup.find_all('a', href=True):
        href = a['href'].strip()
        if not href or href.startswith(('#', 'javascript:')):
            continue
        abs_url = urljoin(url, href)
        text = a.get_text(separator=' ', strip=True)
        results.append({'href': href, 'abs_url': abs_url, 'text': text})
    return results

def _same_domain(base_url: str, target_url: str) -> bool:
    return urlparse(base_url).netloc == urlparse(target_url).netloc

def _extract_pdf_text(pdf_url: str, max_pages: int = 10) -> str:
    r = requests.get(pdf_url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return '\n'.join(p.extract_text() or '' for p in reader.pages[:max_pages]).strip()


In [ ]:
# Cell 6 - Disk Cache

PAGE_CACHE_FILE = '/content/page_cache.json'
_page_cache: dict = {}

def _load_page_cache():
    global _page_cache
    if os.path.exists(PAGE_CACHE_FILE):
        with open(PAGE_CACHE_FILE) as f:
            _page_cache = json.load(f)

def _save_page_cache():
    with open(PAGE_CACHE_FILE, 'w') as f:
        json.dump(_page_cache, f, indent=2)

def cached_page(url: str) -> str:
    if url not in _page_cache:
        print(f'    [HTTP  ] {url}')
        _page_cache[url] = _raw_fetch_text(url)
        _save_page_cache()
    else:
        print(f'    [cache ] {url}')
    return _page_cache[url]

def _sitemap_path(domain: str) -> str:
    return f"/content/sitemap_{domain.replace('.', '_')}.json"

def _load_sitemap(domain: str) -> Optional[dict]:
    path = _sitemap_path(domain)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

def _save_sitemap(domain: str, sitemap: dict):
    with open(_sitemap_path(domain), 'w') as f:
        json.dump(sitemap, f, indent=2)

def clear_cache():
    global _page_cache, _college_url_cache
    _page_cache = {}
    _college_url_cache = {}
    files_to_remove = [PAGE_CACHE_FILE, COLLEGE_URL_CACHE_FILE] + [
        os.path.join('/content', f)
        for f in os.listdir('/content')
        if f.startswith('sitemap_') and f.endswith('.json')
    ]
    for f in files_to_remove:
        if os.path.exists(f):
            os.remove(f)
    print('All caches cleared.')

_load_page_cache()


In [ ]:
# Cell 7 - Pydantic Schemas

class SitemapEntry(BaseModel):
    topic: str = Field(
        description='Topic category e.g. transport, placements, hostel, academics, fees, admissions, facilities, research, about, events, academic_calendar'
    )
    label: str = Field(description='Human-readable label for this link e.g. Bus Routes Page')
    url:   str = Field(description='Full absolute URL')
    kind:  str = Field(description='Either page or pdf')

class CollegeSitemap(BaseModel):
    entries: List[SitemapEntry] = Field(description='All classified links from the college site')

class CollegeLocation(BaseModel):
    address:   str = Field(description='Full street address of the college')
    city:      str = Field(description='City where the college is located')
    state:     str = Field(description='State or province')
    country:   str = Field(description='Country')
    pincode:   str = Field(description='PIN code or ZIP code, empty string if not found')
    map_query: str = Field(description='A Google Maps search query string for this address, e.g. "ANITS College Visakhapatnam Andhra Pradesh"')

class CollegeContact(BaseModel):
    emails:   List[str] = Field(description='List of email addresses found on the college website')
    phones:   List[str] = Field(description='List of phone/fax numbers found on the college website')
    website:  str = Field(description='Official website URL of the college')
    social:   List[str] = Field(description='List of social media profile URLs (LinkedIn, Twitter, Facebook, YouTube, Instagram)')
    summary:  str = Field(description='One-sentence summary of who to contact for admissions enquiries')


# ── NEW: Search Result Schema ─────────────────────────────────────────────────

# Canonical topic categories used across the whole system
SEARCH_CATEGORIES = [
    'hostel',
    'transport',
    'academic_calendar',
    'fees',
    'placements',
    'admissions',
    'facilities',
    'research',
    'events',
    'academics',
    'about',
    'contact',
]

# Human-friendly display names + emoji for each category
CATEGORY_META = {
    'hostel':            {'label': '🏠 Hostel',             'query': 'What hostel facilities are available?'},
    'transport':         {'label': '🚌 Transport',           'query': 'What are the bus/transport routes and timings?'},
    'academic_calendar': {'label': '📅 Academic Calendar',   'query': 'What is the academic calendar and important dates?'},
    'fees':              {'label': '💰 Fees',                'query': 'What is the fee structure?'},
    'placements':        {'label': '💼 Placements',          'query': 'What are the placement statistics and top recruiters?'},
    'admissions':        {'label': '📝 Admissions',          'query': 'What are the admission requirements and process?'},
    'facilities':        {'label': '🏛️ Facilities',          'query': 'What campus facilities are available?'},
    'research':          {'label': '🔬 Research',            'query': 'What research programs and publications are available?'},
    'events':            {'label': '🎉 Events',              'query': 'What upcoming events and fests are there?'},
    'academics':         {'label': '📚 Academics',           'query': 'What academic programs and courses are offered?'},
    'about':             {'label': 'ℹ️ About',               'query': 'Tell me about this college.'},
    'contact':           {'label': '📞 Contact',             'query': 'What are the contact details?'},
}


In [ ]:
# Cell 8 - Sitemap Builder

sitemap_parser = PydanticOutputParser(pydantic_object=CollegeSitemap)

sitemap_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'You are building a topic map of a college website.\n'
        'Given a list of links (anchor text | URL), classify each informational link into a topic.\n'
        'Topics: transport, placements, hostel, academics, fees, admissions, facilities, research, about, events, academic_calendar, contact, other.\n'
        'IMPORTANT: classify academic calendar, exam schedules, and semester dates under academic_calendar.\n'
        'Skip: navigation links, login, search, social media, external sites, duplicate entries.\n'
        'For PDFs set kind=pdf, for pages set kind=page.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Base URL: {base_url}\n\n'
        'Links found across the site (anchor text | full URL):\n'
        '{links_text}'
    )
])

sitemap_chain = sitemap_prompt | llm | sitemap_parser


def build_sitemap(college_name: str, base_url: str = '') -> str:
    """
    Crawl the college site 2 levels deep, classify all links into a topic map,
    and save to disk. Run ONCE per college before asking questions.
    If base_url is not provided, it is resolved automatically from the college name.
    """
    if not base_url:
        base_url = resolve_college_url(college_name)
        if not base_url:
            return f'Could not find the official website for "{college_name}". Please check the name and try again.'
    domain = urlparse(base_url).netloc
    print(f'\nBuilding sitemap for {college_name} ({domain})...')

    print('  [Level 0] Fetching homepage links...')
    try:
        homepage_links = _raw_fetch_links(base_url)
    except Exception as e:
        return f'Error fetching homepage: {e}'

    all_links = list(homepage_links)
    visited = {base_url}
    sub_pages = [
        lnk for lnk in homepage_links
        if _same_domain(base_url, lnk['abs_url'])
        and not lnk['abs_url'].lower().endswith('.pdf')
        and lnk['abs_url'] not in visited
    ][:25]

    for lnk in sub_pages:
        sub_url = lnk['abs_url']
        visited.add(sub_url)
        try:
            sub_links = _raw_fetch_links(sub_url)
            all_links.extend(sub_links)
            print(f'    [Level 1] {sub_url} -> {len(sub_links)} links')
        except Exception:
            pass

    seen_urls = set()
    unique_links = []
    for lnk in all_links:
        abs_url = lnk['abs_url']
        if abs_url in seen_urls or not _same_domain(base_url, abs_url):
            continue
        seen_urls.add(abs_url)
        label = lnk['text'] or abs_url.split('/')[-1] or abs_url
        unique_links.append({'label': label, 'url': abs_url})

    print(f'  {len(unique_links)} unique links collected. Classifying with LLM...')

    links_text = '\n'.join(
        f"{lnk['label']} | {lnk['url']}"
        for lnk in unique_links[:80]
    )
    try:
        result = sitemap_chain.invoke({
            'college_name': college_name,
            'base_url': base_url,
            'links_text': links_text,
            'format_instructions': sitemap_parser.get_format_instructions()
        })
    except Exception as e:
        return f'LLM classification failed: {e}'

    topics: dict = {}
    for entry in result.entries:
        t = entry.topic.lower()
        if t not in topics:
            topics[t] = []
        topics[t].append({'label': entry.label, 'url': entry.url, 'kind': entry.kind})

    sitemap = {
        'college_name': college_name,
        'base_url': base_url,
        'built_at': datetime.now().isoformat(),
        'topics': topics
    }

    _save_sitemap(domain, sitemap)

    summary = f'Sitemap built for {college_name}.\n'
    summary += f'Topics found: {list(topics.keys())}\n'
    summary += f'Total entries: {sum(len(v) for v in topics.values())}'
    print(summary)
    return summary


In [ ]:
# Cell 9 - Tool Registry

_active_sitemap: dict = {}


@tool
def lookup_sitemap(topic: str) -> str:
    """
    Look up the pre-built college sitemap to find pages and PDFs for a given topic.
    ALWAYS call this tool FIRST. Do not call fetch_page before calling this.
    Input: a single topic word such as transport, placements, hostel, fees, academics,
           academic_calendar, admissions, facilities, research, events.
    Returns: labelled URLs classified under that topic, or the full sitemap if no match.
    """
    if not _active_sitemap:
        return (
            'No sitemap loaded. Navigate manually: '
            'call get_page_links on the base URL to discover the site structure.'
        )
    topics = _active_sitemap.get('topics', {})
    topic_lower = topic.lower().replace(' ', '_')
    matched = []
    for t, entries in topics.items():
        if topic_lower in t or t in topic_lower:
            for e in entries:
                matched.append(f"[{e['kind'].upper()}] {e['label']} -> {e['url']}")
    if matched:
        return f"Found {len(matched)} entries for '{topic}':\n" + '\n'.join(matched)
    lines = [f"No match for '{topic}'. Full sitemap:"]
    for t, entries in topics.items():
        lines.append(f'\n[{t.upper()}]')
        for e in entries[:5]:
            lines.append(f"  {e['kind'].upper()} | {e['label']} -> {e['url']}")
    return '\n'.join(lines)


@tool
def fetch_page(url: str) -> str:
    """
    Fetch the visible text content of a college webpage.
    Use this after lookup_sitemap returns a relevant page URL.
    Results are cached - repeated calls to the same URL are free.
    Returns up to 4000 characters of page text.
    """
    try:
        return cached_page(url)[:4000]
    except Exception as e:
        return f'Error fetching {url}: {e}'


@tool
def get_page_links(url: str) -> str:
    """
    Get all links on a page as Anchor Text -> URL pairs.
    Use this to discover deeper pages or find PDF links the sitemap may have missed.
    Only returns links within the same college domain.
    Returns up to 40 links.
    """
    try:
        links = _raw_fetch_links(url)
        lines = [
            f"{lnk['text'] or '(no text)'} -> {lnk['abs_url']}"
            for lnk in links
            if _same_domain(url, lnk['abs_url'])
        ]
        lines = list(dict.fromkeys(lines))[:40]
        return '\n'.join(lines) if lines else 'No same-domain links found on this page.'
    except Exception as e:
        return f'Error getting links from {url}: {e}'


@tool
def read_pdf(url: str) -> str:
    """
    Download and extract text from a PDF document.
    Use this when lookup_sitemap or get_page_links reveals a PDF that likely
    contains the answer such as bus schedules, placement reports, fee structures.
    Returns up to 3000 characters of extracted text.
    """
    try:
        text = _extract_pdf_text(url)
        return text[:3000] if text else 'PDF appears to be empty or image-only.'
    except Exception as e:
        return f'Error reading PDF {url}: {e}'


@tool
def search_web(query: str) -> str:
    """
    Search the web using DuckDuckGo as a fallback when the college website
    does not contain enough information to answer the query.
    Use this ONLY after lookup_sitemap, fetch_page, and get_page_links have
    failed to return relevant results, or when the question asks for
    general/comparative information not found on the college site.
    Input: a concise search query, e.g. 'ANITS college placement statistics 2024'.
    Returns: up to 5 search results with title, URL, and a short snippet.
    """
    try:
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append(
                    f"Title   : {r.get('title', 'N/A')}\n"
                    f"URL     : {r.get('href', 'N/A')}\n"
                    f"Snippet : {r.get('body', 'N/A')}\n"
                )
        if not results:
            return 'No results found for that query.'
        return f"Web search results for '{query}':\n\n" + '\n'.join(results)
    except Exception as e:
        return f'Web search failed: {e}'


# ── NEW: search_topic tool ───────────────────────────────────────────────────

@tool
def search_topic(topic: str) -> str:
    """
    Return all sitemap entries for a specific topic category AND fetch the
    first matching page/PDF to give an immediate summary of available info.
    Supported topics: hostel, transport, academic_calendar, fees, placements,
    admissions, facilities, research, events, academics, about, contact.
    This is the recommended first tool when the user clicks a category button
    (Hostel, Transport, Academic Calendar, etc.).
    Returns: list of links + first-page preview text (up to 2000 chars).
    """
    if not _active_sitemap:
        return 'No sitemap loaded. Build the sitemap first.'

    topics_map = _active_sitemap.get('topics', {})
    topic_key  = topic.lower().replace(' ', '_')

    # Gather matching entries — allow partial match
    matched_entries = []
    for t, entries in topics_map.items():
        if topic_key in t or t in topic_key:
            matched_entries.extend(entries)

    if not matched_entries:
        return (
            f"No sitemap entries found for topic '{topic}'. "
            f"Available topics: {list(topics_map.keys())}. "
            "Try search_web as a fallback."
        )

    lines = [f"=== {topic.upper()} — {len(matched_entries)} sitemap entries ==="]
    for e in matched_entries[:10]:
        lines.append(f"  [{e['kind'].upper()}] {e['label']} -> {e['url']}")

    # Auto-fetch the first page entry for an immediate content preview
    first_page = next((e for e in matched_entries if e['kind'] == 'page'), None)
    first_pdf  = next((e for e in matched_entries if e['kind'] == 'pdf'),  None)
    preview    = ''

    if first_page:
        try:
            content = cached_page(first_page['url'])[:2000]
            preview = f"\n--- Preview: {first_page['label']} ---\n{content}"
        except Exception as e:
            preview = f'\n(Could not fetch preview: {e})'
    elif first_pdf:
        try:
            content = _extract_pdf_text(first_pdf['url'])[:2000]
            preview = f"\n--- PDF Preview: {first_pdf['label']} ---\n{content}"
        except Exception as e:
            preview = f'\n(Could not fetch PDF preview: {e})'

    return '\n'.join(lines) + preview


In [ ]:
# Cell 9b - College URL Resolver

COLLEGE_URL_CACHE_FILE = '/content/college_url_cache.json'
_college_url_cache: dict = {}

def _load_college_url_cache():
    global _college_url_cache
    if os.path.exists(COLLEGE_URL_CACHE_FILE):
        with open(COLLEGE_URL_CACHE_FILE) as f:
            _college_url_cache = json.load(f)

def _save_college_url_cache():
    with open(COLLEGE_URL_CACHE_FILE, 'w') as f:
        json.dump(_college_url_cache, f, indent=2)

def resolve_college_url(college_name: str) -> str:
    key = college_name.strip().lower()
    if key in _college_url_cache:
        print(f'  [URL cache] {college_name} -> {_college_url_cache[key]}')
        return _college_url_cache[key]
    print(f'  [URL lookup] Searching for official site of: {college_name}')
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(f'{college_name} official college website', max_results=5))
        SKIP_DOMAINS = ('wikipedia', 'facebook', 'linkedin', 'justdial',
                        'shiksha', 'collegedunia', 'careers360', 'instagram',
                        'twitter', 'youtube', 'indiamart', 'quora')
        for r in results:
            url = r.get('href', '')
            if url and not any(skip in url for skip in SKIP_DOMAINS):
                parsed = urlparse(url)
                base_url = f'{parsed.scheme}://{parsed.netloc}/'
                _college_url_cache[key] = base_url
                _save_college_url_cache()
                print(f'  [URL lookup] Found: {base_url}')
                return base_url
        fallback = results[0].get('href', '') if results else ''
        if fallback:
            parsed = urlparse(fallback)
            base_url = f'{parsed.scheme}://{parsed.netloc}/'
            _college_url_cache[key] = base_url
            _save_college_url_cache()
            print(f'  [URL lookup] Fallback: {base_url}')
            return base_url
    except Exception as e:
        print(f'  [URL lookup] Error: {e}')
    return ''

_load_college_url_cache()


In [ ]:
# Cell 9c - Location Tool + Contact Info Tool

# ── Location ────────────────────────────────────────────────────────────────

location_parser = PydanticOutputParser(pydantic_object=CollegeLocation)

location_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'Extract the physical address of the college from the provided webpage text.\n'
        'Look for address, location, contact us, reach us sections.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Page text:\n'
        '{page_text}'
    )
])

location_chain = location_prompt | llm | location_parser


def fetch_college_location(college_name: str, base_url: str) -> dict:
    candidates = []
    if _active_sitemap:
        topics = _active_sitemap.get('topics', {})
        for topic in ('about', 'contact', 'facilities', 'other'):
            for entry in topics.get(topic, [])[:2]:
                if entry['kind'] == 'page':
                    candidates.append(entry['url'])
    candidates.append(base_url)

    combined_text = ''
    for url in candidates[:3]:
        try:
            combined_text += cached_page(url)[:2000] + '\n'
        except Exception:
            pass

    if not combined_text.strip():
        return {'address': 'Not found', 'city': '', 'state': '',
                'country': '', 'pincode': '', 'map_query': college_name}

    try:
        result = location_chain.invoke({
            'college_name': college_name,
            'page_text': combined_text[:4000],
            'format_instructions': location_parser.get_format_instructions()
        })
        return result.dict()
    except Exception as e:
        return {'address': f'Could not extract: {e}', 'city': '', 'state': '',
                'country': '', 'pincode': '', 'map_query': college_name}


@tool
def get_college_location(dummy: str = '') -> str:
    """
    Fetch the physical location / address of the college currently being queried.
    Use this when the student asks where the college is located, its address, or
    how to reach it.
    Returns a formatted address string.
    """
    if not _active_sitemap:
        return 'No active college loaded. Please ensure build_sitemap() was called first.'
    college_name = _active_sitemap.get('college_name', 'Unknown')
    base_url     = _active_sitemap.get('base_url', '')
    loc = fetch_college_location(college_name, base_url)
    parts = [loc['address']]
    if loc['city']:    parts.append(loc['city'])
    if loc['state']:   parts.append(loc['state'])
    if loc['pincode']: parts.append(loc['pincode'])
    if loc['country']: parts.append(loc['country'])
    return ', '.join(p for p in parts if p)


# ── Contact Info ─────────────────────────────────────────────────────────────

contact_parser = PydanticOutputParser(pydantic_object=CollegeContact)

contact_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'Extract all contact details for the college from the provided webpage text.\n'
        'Include email addresses, phone/fax numbers, social media links, and the official website.\n'
        'For the summary field, write one sentence saying who to contact for admissions enquiries.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Pre-extracted emails (regex): {raw_emails}\n'
        'Pre-extracted phones (regex): {raw_phones}\n'
        'Page text:\n'
        '{page_text}'
    )
])

contact_chain = contact_prompt | llm | contact_parser


def _regex_extract_contacts(text: str) -> dict:
    """Fast regex pre-scan for emails and phone numbers."""
    emails = list(dict.fromkeys(
        re.findall(r'[\w.+-]+@[\w-]+\.[\w.-]+', text)
    ))[:10]
    phones = list(dict.fromkeys(
        re.findall(r'[\+\(]?[0-9][0-9 .\-\(\)]{7,}[0-9]', text)
    ))[:10]
    return {'emails': emails, 'phones': phones}


@tool
def get_contact_info(dummy: str = '') -> str:
    """
    Extract contact details for the college currently being queried.
    Returns email addresses, phone/fax numbers, social media links, and
    a brief summary of who to contact for admissions.
    Use this when a student asks for email, phone, contact details, or how
    to get in touch with the college.
    """
    if not _active_sitemap:
        return 'No active college loaded. Please ensure build_sitemap() was called first.'

    college_name = _active_sitemap.get('college_name', 'Unknown')
    base_url     = _active_sitemap.get('base_url', '')

    candidates = []
    if _active_sitemap:
        topics = _active_sitemap.get('topics', {})
        for topic in ('contact', 'about', 'admissions', 'other'):
            for entry in topics.get(topic, [])[:2]:
                if entry['kind'] == 'page':
                    candidates.append(entry['url'])
    candidates.append(base_url)

    combined_text = ''
    for url in candidates[:3]:
        try:
            combined_text += cached_page(url)[:2000] + '\n'
        except Exception:
            pass

    if not combined_text.strip():
        return 'Could not fetch any pages to extract contact info.'

    raw = _regex_extract_contacts(combined_text)

    try:
        result = contact_chain.invoke({
            'college_name': college_name,
            'raw_emails': ', '.join(raw['emails']) or 'none found',
            'raw_phones': ', '.join(raw['phones']) or 'none found',
            'page_text': combined_text[:4000],
            'format_instructions': contact_parser.get_format_instructions()
        })
        lines = [f'**Contact Info for {college_name}**\n']
        if result.emails:
            lines.append('**Email(s):** ' + ', '.join(result.emails))
        if result.phones:
            lines.append('**Phone(s):** ' + ', '.join(result.phones))
        if result.website:
            lines.append(f'**Website:** {result.website}')
        if result.social:
            lines.append('**Social:** ' + ', '.join(result.social))
        if result.summary:
            lines.append(f'\n_{result.summary}_')
        return '\n'.join(lines)
    except Exception as e:
        if raw['emails'] or raw['phones']:
            return (
                f"Emails : {', '.join(raw['emails']) or 'not found'}\n"
                f"Phones : {', '.join(raw['phones']) or 'not found'}"
            )
        return f'Could not extract contact info: {e}'


# Final tool registry — search_topic added
TOOLS = [lookup_sitemap, fetch_page, get_page_links, read_pdf, search_web,
         search_topic, get_college_location, get_contact_info]
print(f'Tools registered: {[t.name for t in TOOLS]}')


Tools registered: ['lookup_sitemap', 'fetch_page', 'get_page_links', 'read_pdf', 'search_web', 'search_topic', 'get_college_location', 'get_contact_info']


In [ ]:
# Cell 10 - ReAct Agent

AGENT_SYSTEM_PROMPT = (
    'You are a Campus Query Agent that answers student questions about colleges and universities.\n'
    'You have a pre-built sitemap and eight tools to navigate the college website.\n'
    '\n'
    'TOOL ORDER - always follow this sequence:\n'
    '1. search_topic(topic) - use this FIRST when the query maps to a known category\n'
    '   (hostel, transport, academic_calendar, fees, placements, admissions, facilities, research, events, academics).\n'
    '   This returns all sitemap links AND a page preview in one call.\n'
    '2. lookup_sitemap(topic) - use for freeform topics not covered by search_topic.\n'
    '3. fetch_page(url) - fetch the most relevant page returned by search_topic or lookup_sitemap.\n'
    '4. get_page_links(url) - if you need to go deeper or find a PDF, get links from that page.\n'
    '5. read_pdf(url) - if a PDF is found that likely has the answer, read it.\n'
    '6. get_college_location() - use this when the query is about the college address or location.\n'
    '7. get_contact_info() - use this when the student asks for email, phone, or contact details.\n'
    '8. search_web(query) - FALLBACK ONLY. Use this if steps 1-7 did not return useful information,\n'
    '   or if the question asks for general/comparative facts not on the college site.\n'
    '   Always include the college name in the search query for context.\n'
    '9. Answer with a clear, concise markdown response once you have enough information.\n'
    '\n'
    'RULES:\n'
    '- Never guess. Only answer from what the tools actually return.\n'
    '- If no sitemap match, call get_page_links on the college base URL to navigate manually.\n'
    '- After 4 tool calls with no relevant result from the site, use search_web as a fallback.\n'
    '- Always prefer information from the official college website over web search results.\n'
    '- Format your final answer in clean markdown. Cite sources (URLs) where possible.\n'
)

agent_graph = create_react_agent(llm, TOOLS)
print('ReAct agent ready.')


In [ ]:
# Cell 11 - Pipeline

def create_query(college_name: str, query: str, url: str = '') -> str:
    global _active_sitemap

    if not url:
        url = resolve_college_url(college_name)
        if not url:
            return f'Could not find the official website for "{college_name}". Please check the name and try again.'

    print(f"\n{'='*60}")
    print(f'Query   : {query}')
    print(f'College : {college_name} ({url})')
    print('='*60)

    domain = urlparse(url).netloc
    sitemap = _load_sitemap(domain)
    if sitemap:
        _active_sitemap = sitemap
        print(f"  Sitemap loaded: topics = {list(sitemap['topics'].keys())}")
    else:
        _active_sitemap = {'college_name': college_name, 'base_url': url, 'topics': {}}
        print('  No sitemap found. Build one first with build_sitemap() for best results.')
        print('  Agent will navigate manually using get_page_links.')

    user_message = (
        f'College: {college_name}\n'
        f'Website: {url}\n'
        f'Query: {query}'
    )

    print('\n[Agent running...]')
    result = agent_graph.invoke({
        'messages': [
            SystemMessage(content=AGENT_SYSTEM_PROMPT),
            HumanMessage(content=user_message)
        ]
    })

    print('\n[Agent trace]')
    tool_call_count = 0
    for msg in result['messages'][2:]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_call_count += 1
                print(f'  [{tool_call_count}] CALL   {tc["name"]}({tc["args"]})')
        elif type(msg).__name__ == 'ToolMessage':
            preview = (msg.content or '')[:100].replace('\n', ' ')
            print(f'      RESULT {preview}...')

    print(f'\n  Total tool calls: {tool_call_count}')
    return result['messages'][-1].content


In [ ]:
# Cell 12 - Build the sitemap (run once per college)
status = build_sitemap('ANITS')
print(status)


In [ ]:
# Cell 13 - Ask a question
answer = create_query('ANITS', 'What are the placement statistics?')
display(Markdown(answer))


In [ ]:
# Cell 14 - Gradio UI  (Smart Search tab added)
import gradio as gr
import urllib.parse


def gradio_build_sitemap(college_name: str) -> None:
    if not college_name.strip():
        print('Please enter a college name.')
        return
    result = build_sitemap(college_name.strip())
    print(result)


def gradio_ask(college_name: str, query: str) -> str:
    if not college_name.strip() or not query.strip():
        return 'Please enter both a college name and a query.'
    return create_query(college_name.strip(), query.strip())


def gradio_get_location(college_name: str):
    if not college_name.strip():
        return 'Please enter a college name.', ''
    global _active_sitemap
    url = resolve_college_url(college_name.strip())
    if url:
        domain = urlparse(url).netloc
        sitemap = _load_sitemap(domain)
        _active_sitemap = sitemap if sitemap else {'college_name': college_name.strip(), 'base_url': url, 'topics': {}}
    else:
        url = ''
        _active_sitemap = {'college_name': college_name.strip(), 'base_url': '', 'topics': {}}

    loc = fetch_college_location(college_name.strip(), url)
    parts = [loc['address']]
    if loc['city']:    parts.append(loc['city'])
    if loc['state']:   parts.append(loc['state'])
    if loc['pincode']: parts.append(loc['pincode'])
    if loc['country']: parts.append(loc['country'])
    address_str = ', '.join(p for p in parts if p and p != 'Not found')

    map_query    = loc.get('map_query') or college_name.strip()
    encoded_query = urllib.parse.quote(map_query)
    map_html = (
        f'<iframe '
        f'width="100%" height="350" style="border:0;border-radius:8px;" '
        f'loading="lazy" allowfullscreen '
        f'src="https://maps.google.com/maps?q={encoded_query}&output=embed">'
        f'</iframe>'
    )
    return address_str or 'Address not found on website.', map_html


def gradio_get_contact(college_name: str) -> str:
    if not college_name.strip():
        return 'Please enter a college name.'
    global _active_sitemap
    url = resolve_college_url(college_name.strip())
    if url:
        domain = urlparse(url).netloc
        sitemap = _load_sitemap(domain)
        _active_sitemap = sitemap if sitemap else {'college_name': college_name.strip(), 'base_url': url, 'topics': {}}
    else:
        _active_sitemap = {'college_name': college_name.strip(), 'base_url': '', 'topics': {}}
    return get_contact_info('')


# ── NEW: Smart Search helpers ─────────────────────────────────────────────────

def gradio_smart_search(college_name: str, category: str) -> str:
    """
    Called when user clicks a category button in the Smart Search tab.
    Loads the sitemap then runs the agent with a category-specific query.
    """
    if not college_name.strip():
        return 'Please enter a college name at the top first.'
    meta  = CATEGORY_META.get(category, {})
    query = meta.get('query', f'Tell me about {category} at this college.')
    return create_query(college_name.strip(), query)


def gradio_smart_search_custom(college_name: str, custom_query: str) -> str:
    """Free-text search from the Smart Search tab."""
    if not college_name.strip() or not custom_query.strip():
        return 'Please enter both a college name and a search query.'
    return create_query(college_name.strip(), custom_query.strip())


def _make_category_click(cat_key: str):
    """Return a Gradio-compatible function bound to a specific category."""
    def _fn(college_name, _placeholder=''):
        return gradio_smart_search(college_name, cat_key)
    return _fn


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks(title='Campus Query Agent', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎓 Campus Query Agent')
    gr.Markdown(
        '**Step 1:** Enter a college name and build its sitemap once (1 LLM call, saved to disk).  \n'
        '**Step 2:** Use any tab below — Ask a custom question, browse by category, view location, or get contact info.'
    )

    college_name = gr.Textbox(label='College Name', placeholder='e.g. ANITS or IIT Bombay')

    with gr.Row():
        build_btn = gr.Button('Build Sitemap', variant='secondary')
        clear_btn = gr.Button('Clear Cache',   variant='secondary')

    gr.Markdown('---')

    with gr.Tabs():

        # ── Tab 1: Smart Search (NEW) ─────────────────────────────────────────
        with gr.TabItem('🔍 Smart Search'):
            gr.Markdown(
                '### Browse by Category\n'
                'Click any button to instantly search that topic on the college website.'
            )

            search_output = gr.Markdown(value='_Search results will appear here..._')

            # Row 1
            with gr.Row():
                btn_hostel   = gr.Button('🏠 Hostel')
                btn_transport = gr.Button('🚌 Transport')
                btn_calendar  = gr.Button('📅 Academic Calendar')
                btn_fees      = gr.Button('💰 Fees')

            # Row 2
            with gr.Row():
                btn_placements  = gr.Button('💼 Placements')
                btn_admissions  = gr.Button('📝 Admissions')
                btn_facilities  = gr.Button('🏛️ Facilities')
                btn_research    = gr.Button('🔬 Research')

            # Row 3
            with gr.Row():
                btn_events    = gr.Button('🎉 Events')
                btn_academics = gr.Button('📚 Academics')
                btn_about     = gr.Button('ℹ️ About')
                btn_contact2  = gr.Button('📞 Contact')

            gr.Markdown('#### Or type your own search:')
            with gr.Row():
                custom_search_box = gr.Textbox(
                    label='',
                    placeholder='e.g. scholarships, library timings, sports facilities...',
                    scale=4
                )
                custom_search_btn = gr.Button('Search', variant='primary', scale=1)

            # Wire category buttons
            for btn, cat in [
                (btn_hostel,    'hostel'),
                (btn_transport, 'transport'),
                (btn_calendar,  'academic_calendar'),
                (btn_fees,      'fees'),
                (btn_placements,'placements'),
                (btn_admissions,'admissions'),
                (btn_facilities,'facilities'),
                (btn_research,  'research'),
                (btn_events,    'events'),
                (btn_academics, 'academics'),
                (btn_about,     'about'),
                (btn_contact2,  'contact'),
            ]:
                btn.click(
                    fn=_make_category_click(cat),
                    inputs=[college_name],
                    outputs=search_output
                )

            # Wire custom search
            custom_search_btn.click(
                fn=gradio_smart_search_custom,
                inputs=[college_name, custom_search_box],
                outputs=search_output
            )

        # ── Tab 2: Ask a Query ────────────────────────────────────────────────
        with gr.TabItem('💬 Ask a Query'):
            user_query = gr.Textbox(
                label='Your Query',
                placeholder='e.g. What are the bus routes?'
            )
            ask_btn = gr.Button('Ask Agent', variant='primary')
            output  = gr.Markdown(value='_Your answer will appear here..._')

            gr.Examples(
                examples=[
                    ['ANITS',      'What are the transportation routes?'],
                    ['ANITS',      'What are the placement statistics?'],
                    ['ANITS',      'What hostel facilities are available?'],
                    ['ANITS',      'What is the academic calendar?'],
                    ['IIT Bombay', 'What are the admission requirements?'],
                    ['ANITS',      'What is the email address for admissions?'],
                ],
                inputs=[college_name, user_query]
            )

            ask_btn.click(fn=gradio_ask, inputs=[college_name, user_query], outputs=output)

        # ── Tab 3: College Location ───────────────────────────────────────────
        with gr.TabItem('📍 College Location'):
            gr.Markdown(
                '### Find the College Location\n'
                'Extracts the physical address from the college website and shows it on a map.'
            )
            location_btn = gr.Button('Get Location', variant='primary')
            address_out  = gr.Textbox(
                label='Extracted Address',
                interactive=False,
                placeholder='Address will appear here...'
            )
            map_out = gr.HTML(label='Map')

            location_btn.click(
                fn=gradio_get_location,
                inputs=[college_name],
                outputs=[address_out, map_out]
            )

        # ── Tab 4: Contact Info ───────────────────────────────────────────────
        with gr.TabItem('📞 Contact Info'):
            gr.Markdown(
                '### Get College Contact Details\n'
                'Extracts email addresses, phone numbers, and social media links from the college website.'
            )
            contact_btn = gr.Button('Get Contact Info', variant='primary')
            contact_out = gr.Markdown(value='_Contact details will appear here..._')

            contact_btn.click(
                fn=gradio_get_contact,
                inputs=[college_name],
                outputs=contact_out
            )

    build_btn.click(fn=gradio_build_sitemap, inputs=[college_name])
    clear_btn.click(fn=lambda: clear_cache())

if __name__ == '__main__':
    demo.launch()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


* Running on public URL: https://69445db8c61a2a74ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag